# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [1]:
import pandas as pd

---
## 2. Load data

In [2]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [3]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [10]:
price_df = pd.read_csv('price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'price_data.csv'

In [12]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

In [13]:
# Calculate the range of off-peak variable prices for each customer

offpeak_var_range = monthly_price_by_id.groupby('id')['price_off_peak_var'].agg(
lambda x: x.max() - x.min()
).reset_index()

offpeak_var_range = offpeak_var_range.rename(
columns={'price_off_peak_var': 'offpeak_var_price_range'}
)

offpeak_var_range.head()

,id,offpeak_var_price_range
0,0002203ffbb812588b632b9e628cc38d,0.008161
1,0004351ebdd665e6ee664792efc4fd13,0.004462
2,0010bcc39e42b3c2131ed2ce55246e3c,0.054905
3,0010ee3855fdea87602a5b7aba8e42de,0.010018
4,00114d74e963e47177db89bc70108537,0.004462


In [8]:
price_df = pd.read_csv('price_data (1) (1).csv')

In [14]:
# Calculate the average off-peak fixed price for each customer

offpeak_fix_avg = monthly_price_by_id.groupby('id')['price_off_peak_fix'].mean().reset_index()

offpeak_fix_avg = offpeak_fix_avg.rename(
columns={'price_off_peak_fix': 'offpeak_fix_price_avg'}
)

offpeak_fix_avg.head()

,id,offpeak_fix_price_avg
0,0002203ffbb812588b632b9e628cc38d,40.701732
1,0004351ebdd665e6ee664792efc4fd13,44.385450
2,0010bcc39e42b3c2131ed2ce55246e3c,45.319710
3,0010ee3855fdea87602a5b7aba8e42de,40.647427
4,00114d74e963e47177db89bc70108537,44.266930


In [15]:
# Combine our new features into one customer-level table

new_features = offpeak_var_range.merge(
offpeak_fix_avg,
on='id',
how='left'
)

new_features.head()

,id,offpeak_var_price_range,offpeak_fix_price_avg
0,0002203ffbb812588b632b9e628cc38d,0.008161,40.701732
1,0004351ebdd665e6ee664792efc4fd13,0.004462,44.385450
2,0010bcc39e42b3c2131ed2ce55246e3c,0.054905,45.319710
3,0010ee3855fdea87602a5b7aba8e42de,0.010018,40.647427
4,00114d74e963e47177db89bc70108537,0.004462,44.266930


In [16]:
new_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 16096 entries, 0 to 16095
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       16096 non-null  str    
 1   offpeak_var_price_range  16096 non-null  float64
 2   offpeak_fix_price_avg    16096 non-null  float64
dtypes: float64(2), str(1)
memory usage: 882.3 KB


In [17]:
# Add our new features to the main customer dataset

df = df.merge(new_features, on='id', how='left')

df.head()

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn,offpeak_var_price_range,offpeak_fix_price_avg
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,9.084737e-04,2.086294,99.530517,44.235794,2.086425,9.953056e+01,4.423670e+01,1,0.028554,40.942265
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000000e+00,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000e+00,0,0.005334,44.311375
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000000e+00,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00,0,0.004670,44.385450
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,0.000000e+00,0.000000,0.000000,0.000000,0.000003,0.000000e+00,0.000000e+00,0,0.004547,44.400265
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,4.860000e-10,0.000000,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10,0,0.008161,40.688156


In [18]:
# Calculate the change between last month's consumption and 12-month consumption

df['consumption_change'] = df['cons_last_month'] - (df['cons_12m'] / 12)

df[['id', 'cons_12m', 'cons_last_month', 'consumption_change']].head()


,id,cons_12m,cons_last_month,consumption_change
0,24011ae4ebbe3035111d65fa7c15bc57,0,0,0.000000
1,d29c2c54acc38ff3c0614d0a653813dd,4660,0,-388.333333
2,764c75f661154dac3a6c254cd082ea7d,544,0,-45.333333
3,bba03439a292a1e166f80264c16191cb,1584,0,-132.000000
4,149d57cf92fc41cf94415803a877cb4b,4425,526,157.250000


In [19]:
# Calculate last month's consumption relative to the 12-month total

df['consumption_ratio'] = df['cons_last_month'] / df['cons_12m'].replace(0, 1)

df[['id', 'cons_12m', 'cons_last_month', 'consumption_ratio']].head()

,id,cons_12m,cons_last_month,consumption_ratio
0,24011ae4ebbe3035111d65fa7c15bc57,0,0,0.00000
1,d29c2c54acc38ff3c0614d0a653813dd,4660,0,0.00000
2,764c75f661154dac3a6c254cd082ea7d,544,0,0.00000
3,bba03439a292a1e166f80264c16191cb,1584,0,0.00000
4,149d57cf92fc41cf94415803a877cb4b,4425,526,0.11887


In [20]:
# Add the December-January price difference features to the main dataset

df = df.merge(
diff[['id', 'offpeak_diff_dec_january_energy', 'offpeak_diff_dec_january_power']],
on='id',
how='left'
)

df.head()

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn,offpeak_var_price_range,offpeak_fix_price_avg,consumption_change,consumption_ratio,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,2.086425,9.953056e+01,4.423670e+01,1,0.028554,40.942265,0.000000,0.00000,0.020057,3.700961
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.009485,1.217891e-03,0.000000e+00,0,0.005334,44.311375,-388.333333,0.00000,-0.003767,0.177779
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000e+00,0,0.004670,44.385450,-45.333333,0.00000,-0.004670,0.177779
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,0.000003,0.000000e+00,0.000000e+00,0,0.004547,44.400265,-132.000000,0.00000,-0.004547,0.177779
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,0.000011,2.896760e-06,4.860000e-10,0,0.008161,40.688156,157.250000,0.11887,-0.006192,0.162916


In [21]:
df[['id',
'offpeak_var_price_range',
'offpeak_fix_price_avg',
'consumption_change',
'consumption_ratio',
'offpeak_diff_dec_january_energy',
'offpeak_diff_dec_january_power']].head()

,id,offpeak_var_price_range,offpeak_fix_price_avg,consumption_change,consumption_ratio,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,24011ae4ebbe3035111d65fa7c15bc57,0.028554,40.942265,0.000000,0.00000,0.020057,3.700961
1,d29c2c54acc38ff3c0614d0a653813dd,0.005334,44.311375,-388.333333,0.00000,-0.003767,0.177779
2,764c75f661154dac3a6c254cd082ea7d,0.004670,44.385450,-45.333333,0.00000,-0.004670,0.177779
3,bba03439a292a1e166f80264c16191cb,0.004547,44.400265,-132.000000,0.00000,-0.004547,0.177779
4,149d57cf92fc41cf94415803a877cb4b,0.008161,40.688156,157.250000,0.11887,-0.006192,0.162916
